In [6]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [7]:
from jubilee_controller import JubileeMotionController
import matplotlib.pyplot as plt
import time
import numpy as np 
from pynput import keyboard
import cv2
import lgpio
from datetime import datetime
from email_sender import send_email
from image_processing import detect_circle
from gripper_controller import Gripper
from micropipette_controller import Micropipette
from magnetic_stirrer_controll import Agitador

In [3]:
import lgpio
import time


class GY31:
    def __init__(self, s2=20, s3=21, out=16):
        self.S2 = s2
        self.S3 = s3
        self.OUT = out

        # abre gpiochip
        self.h = lgpio.gpiochip_open(0)

        # configura GPIO
        lgpio.gpio_claim_output(self.h, self.S2)
        lgpio.gpio_claim_output(self.h, self.S3)
        lgpio.gpio_claim_input(self.h, self.OUT)

    def _read_frequency(self, duration=0.1):
        start = time.time()
        count = 0

        last = lgpio.gpio_read(self.h, self.OUT)

        while time.time() - start < duration:
            current = lgpio.gpio_read(self.h, self.OUT)

            # borda de descida
            if last == 1 and current == 0:
                count += 1

            last = current

        return count

    def read_color(self):
        # RED
        lgpio.gpio_write(self.h, self.S2, 0)
        lgpio.gpio_write(self.h, self.S3, 0)
        time.sleep(0.02)
        red = self._read_frequency()

        # BLUE
        lgpio.gpio_write(self.h, self.S2, 0)
        lgpio.gpio_write(self.h, self.S3, 1)
        time.sleep(0.02)
        blue = self._read_frequency()

        # GREEN
        lgpio.gpio_write(self.h, self.S2, 1)
        lgpio.gpio_write(self.h, self.S3, 1)
        time.sleep(0.02)
        green = self._read_frequency()

        return {
            "red": red,
            "green": green,
            "blue": blue
        }

    def save_readings(
        self,
        filename="leituras.txt",
        n=100,
        interval=0.05
    ):
        """
        Faz N leituras e salva em TXT
        """

        with open(filename, "w") as f:

            # cabeçalho
            f.write("timestamp,red,green,blue\n")

            for i in range(n):

                data = self.read_color()

                timestamp = time.time()

                line = (
                    f"{timestamp},"
                    f"{data['red']},"
                    f"{data['green']},"
                    f"{data['blue']}\n"
                )

                f.write(line)

                print(
                    f"[{i+1}/{n}] "
                    f"R={data['red']} "
                    f"G={data['green']} "
                    f"B={data['blue']}"
                )

                time.sleep(interval)

    def close(self):
        lgpio.gpiochip_close(self.h)

In [19]:

sensor = GY31()

try:

    # leitura simples
    print(sensor.read_color())

    # salvar 200 leituras
    sensor.save_readings(
        filename="dados_vermelho.txt",
        n=10,
        interval=0.02
    )

finally:
    sensor.close()

{'red': 794, 'green': 441, 'blue': 1203}
[1/10] R=904 G=414 B=1173
[2/10] R=924 G=439 B=1134
[3/10] R=897 G=403 B=1121
[4/10] R=900 G=416 B=1162
[5/10] R=920 G=411 B=1175
[6/10] R=896 G=394 B=1178
[7/10] R=814 G=378 B=1184
[8/10] R=857 G=382 B=1183
[9/10] R=840 G=379 B=1120
[10/10] R=827 G=363 B=1174


In [8]:
jubilee = JubileeMotionController()

In [10]:
jubilee.home_all(mesh_mode_z=False)
jubilee.protect_tools(on=True)
jubilee.move_xyz_absolute(z=50)

In [12]:
pipeta = Micropipette(jubilee,parking_position_xy=(136.5,16),move_velocity=3000)

In [13]:
pipeta.install()

In [7]:
agitador = Agitador()

Agitador configurado no GPIO6.


In [14]:
jubilee.protect_tools(True)

In [15]:
jubilee.gcode("M208 Z160:320")

'\n'

In [16]:
pipeta.tip = True

In [21]:
altura_segura = 320
posicao_agua = (225,213,320)
posicao_leite = (62.5,345,160)

In [26]:
pipeta.uninstall()

In [24]:
while True:

    pipeta.pipette_liquid(posicao_leite,posicao_agua,100,safe_height=altura_segura,velocidade_dispensacao=300)
    #agitador.ligar()
    time.sleep(30)
    #agitador.desligar()

KeyboardInterrupt: 